# Random Forest Cloud — Entrenamiento en Kaggle Notebooks

**Corre en Kaggle Notebooks** (CPU, ~9h disponibles).

**Dataset requerido:** `mecmt07-features` (subir `features_train.parquet` y `features_test.parquet`).

**Flujo:**
1. Optuna 20 trials: CV AUC penalizado por overfitting (train completo, 307k filas)
2. Refit final con mejores hiperparámetros sobre train completo
3. Métricas + 4 gráficos (Optuna history, ROC, Feature importance, Score distribution)
4. Guardar modelo + metadata → descargar desde Output tab → correr `rfr_cloud_predict.ipynb`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '--quiet'], check=True)
print('Dependencias listas.')

In [ ]:
import subprocess as _sp
print('─' * 50)
print('DETECCIÓN DE GPU / CPU')
print('─' * 50)
_r = _sp.run(['nvidia-smi', '--query-gpu=name,memory.total',
              '--format=csv,noheader'], capture_output=True, text=True)
if _r.returncode == 0:
    print(f'  GPU: {_r.stdout.strip()}')
else:
    print('  Sin GPU — Random Forest usa CPU')
print('─' * 50)
print('  Random Forest es CPU-bound. n_jobs=-1 usa todos los cores disponibles.')
USE_GPU = False

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import sklearn

print(f'numpy   version : {np.__version__}')
print(f'sklearn version : {sklearn.__version__}')
print(f'optuna  version : {optuna.__version__}')
print('Imports OK')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════
SEED     = 42
N_TRIALS = 20
N_FOLDS  = 5

DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

# Matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(SEED)

print('=' * 65)
print('CONFIGURACIÓN')
print('=' * 65)
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')
print(f'  N_TRIALS  : {N_TRIALS}')
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  SEED      : {SEED}')
print(f'  CPU       : n_jobs=-1 (todos los cores)')
print('=' * 65)
for f in ['features_train.parquet', 'features_test.parquet']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "NO encontrado"}')

In [ ]:
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Encodear columnas categóricas (object dtype) → enteros, preservando NaN
cat_cols = [c for c in feature_cols if df[c].dtype == 'object']
if cat_cols:
    print(f'  Columnas categóricas encontradas: {cat_cols}')
    for col in cat_cols:
        cats    = pd.concat([df[col], df_test[col]]).dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df[col]      = df[col].map(mapping)
        df_test[col] = df_test[col].map(mapping)
    print(f'  Encoding completado ✓')
else:
    print('  Sin columnas categóricas')

X           = df[feature_cols].values
y           = df['TARGET'].values
X_test      = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

# Verificar soporte nativo de NaN en sklearn
sklearn_ver = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
NATIVE_NAN  = sklearn_ver >= (1, 4)

if not NATIVE_NAN:
    print('  sklearn < 1.4: imputando NaN con mediana para RF')
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')
    X      = imp.fit_transform(X)
    X_test = imp.transform(X_test)
else:
    print('  sklearn >= 1.4: RF maneja NaN nativamente')

print(f'\nTrain : {X.shape}  | Default rate: {y.mean():.2%}')
print(f'Test  : {X_test.shape}')
print(f'NaNs en X: {np.isnan(X).sum():,}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc, 4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec, 4), Precision=round(prec, 4), F1=round(f1, 4))

print('Funciones auxiliares definidas.')

## 1. Optuna — Búsqueda de Hiperparámetros

In [ ]:
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
trial_stats = {}

def objective_rf(trial):
    n_estimators      = trial.suggest_int('n_estimators',    100, 600)
    max_depth         = trial.suggest_int('max_depth',         4,  25)
    min_samples_split = trial.suggest_int('min_samples_split', 2,  20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf',  1,  10)
    max_features      = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5])

    rf = RandomForestClassifier(
        n_estimators      = n_estimators,
        max_depth         = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf  = min_samples_leaf,
        max_features      = max_features,
        class_weight      = 'balanced',
        n_jobs            = -1,
        random_state      = SEED
    )

    cv_scores = cross_val_score(rf, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_auc    = cv_scores.mean()

    rf.fit(X, y)
    train_auc = roc_auc_score(y, rf.predict_proba(X)[:, 1])

    gap = max(0.0, train_auc - cv_auc)
    trial_stats[trial.number] = {'cv': cv_auc, 'train': train_auc, 'gap': gap}
    return cv_auc - 0.5 * gap

print('Función objetivo definida.')

In [ ]:
def _optuna_callback(study, trial):
    stats  = trial_stats.get(trial.number, {})
    cv     = stats.get('cv', trial.value)
    gap    = stats.get('gap', 0.0)
    marker = ' ◀ best' if trial.number == study.best_trial.number else ''
    print(f'  Trial {trial.number + 1:>2}/{N_TRIALS} │ '
          f'obj={trial.value:.5f}  cv={cv:.5f}  gap={gap:.5f}  │  '
          f'best={study.best_value:.5f}{marker}')

print('=' * 65)
print('OPTUNA — BÚSQUEDA DE HIPERPARÁMETROS (Random Forest)')
print('=' * 65)
print(f'  Trials   : {N_TRIALS}')
print(f'  CV folds : {N_FOLDS}')
print(f'  Datos    : {X.shape[0]:,} filas (train completo)')
print(f'  CPU      : n_jobs=-1')
print('─' * 65)

t_opt    = time.time()
study_rf = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_rf.optimize(objective_rf, n_trials=N_TRIALS,
                  show_progress_bar=False, callbacks=[_optuna_callback])

elapsed_opt = time.time() - t_opt
print('─' * 65)
print(f'Búsqueda finalizada en {elapsed_opt/60:.1f} min')
print(f'\nMejores hiperparámetros:')
for k, v in study_rf.best_params.items():
    print(f'  {k:<22}: {v}')
print(f'\nObjetivo (CV AUC penalizado): {study_rf.best_value:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
trials_nums = [t.number for t in study_rf.trials]
trials_vals = [t.value  for t in study_rf.trials]
best_so_far, cur = [], float('-inf')
for v in trials_vals:
    cur = max(cur, v)
    best_so_far.append(cur)

ax.scatter(trials_nums, trials_vals, alpha=0.6, s=40, color='#27ae60', label='Trial')
ax.plot(trials_nums, best_so_far, color='#e74c3c', lw=2, label='Mejor acumulado')
ax.set_xlabel('Numero de trial')
ax.set_ylabel('Objetivo (CV AUC penalizado)')
ax.set_title('Optuna — Historial de optimizacion (Random Forest)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rfr_optuna_history.png', dpi=120)
plt.show()
print('Grafico guardado: rfr_optuna_history.png')

## 2. Modelo Final — Refit en Train Completo

In [ ]:
best_p = study_rf.best_params

rf_final = RandomForestClassifier(
    n_estimators      = best_p['n_estimators'],
    max_depth         = best_p['max_depth'],
    min_samples_split = best_p['min_samples_split'],
    min_samples_leaf  = best_p['min_samples_leaf'],
    max_features      = best_p['max_features'],
    class_weight      = 'balanced',
    n_jobs            = -1,
    random_state      = SEED
)

print('=' * 65)
print('MODELO FINAL — REFIT EN TRAIN COMPLETO')
print('=' * 65)
print(f'  n_estimators      : {best_p["n_estimators"]}')
print(f'  max_depth         : {best_p["max_depth"]}')
print(f'  min_samples_split : {best_p["min_samples_split"]}')
print(f'  min_samples_leaf  : {best_p["min_samples_leaf"]}')
print(f'  max_features      : {best_p["max_features"]}')
print(f'  Datos             : {X.shape[0]:,} filas (train completo)')
print('─' * 65)

t0 = time.time()
rf_final.fit(X, y)
print(f'  RF entrenado ({time.time() - t0:.0f}s)')

# OOF predictions (N_FOLDS-fold)
print(f'  Calculando OOF predictions ({N_FOLDS}-fold CV)...')
t1 = time.time()
oof_prob = cross_val_predict(
    RandomForestClassifier(**best_p, class_weight='balanced', n_jobs=-1, random_state=SEED),
    X, y,
    cv=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED),
    method='predict_proba', n_jobs=1
)[:, 1]
cv_auc = roc_auc_score(y, oof_prob)
print(f'  OOF listo ({time.time() - t1:.0f}s)')

# Train AUC (in-sample)
y_prob_train = rf_final.predict_proba(X)[:, 1]
train_auc    = roc_auc_score(y, y_prob_train)

print('─' * 65)
print(f'  CV AUC ({N_FOLDS}-fold OOF) : {cv_auc:.5f}')
print(f'  Train AUC (in-sample) : {train_auc:.5f}')
print(f'  Gap                   : {abs(train_auc - cv_auc):.5f}')
print('─' * 65)

## 3. Metricas sobre Train Completo

In [ ]:
metrics_train = compute_metrics(y, y_prob_train, label='RandomForest (train in-sample)')
metrics_oof   = compute_metrics(y, oof_prob,     label='RandomForest (OOF 5-fold)')

for m in [metrics_train, metrics_oof]:
    print(f"\n{m['Model']}")
    print(f"  AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Precision={m['Precision']:.4f}  F1={m['F1']:.4f}")
    print(f"  TP={m['TP']:,}  TN={m['TN']:,}  FP={m['FP']:,}  FN={m['FN']:,}")

display(pd.DataFrame([metrics_train, metrics_oof]).set_index('Model'))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for y_prob, label, color in [
        (y_prob_train, f'Train (AUC={train_auc:.4f})', '#e74c3c'),
        (oof_prob,     f'OOF CV (AUC={cv_auc:.4f})',   '#3498db')]:
    fpr, tpr, _ = roc_curve(y, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=label)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR (1 - Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title('ROC Curve — Random Forest')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rfr_roc_curve.png', dpi=120)
plt.show()
print('Grafico guardado: rfr_roc_curve.png')

## 4. Importancia de Variables

In [ ]:
feat_imp = pd.Series(rf_final.feature_importances_, index=feature_cols)
top15    = feat_imp.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#27ae60' if v >= top15.median() else '#3498db' for v in top15.values]
ax.barh(top15.index, top15.values, color=colors)
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Top-15 Variables — Random Forest')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rfr_feature_importance.png', dpi=120)
plt.show()
print('Grafico guardado: rfr_feature_importance.png')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde   = gaussian_kde(probs, bw_method=0.1)
    xs    = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribucion de scores — Random Forest')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'rfr_score_distribution.png', dpi=120)
plt.show()
print('Grafico guardado: rfr_score_distribution.png')

## 5. Guardar Modelo y Metadata

In [ ]:
model_path  = MODEL_DIR / 'rfr_cloud_best.pkl'
meta_path   = MODEL_DIR / 'rfr_cloud_metadata.json'
trials_path = MODEL_DIR / 'rfr_cloud_optuna_trials.csv'

joblib.dump(rf_final, model_path)

metadata = {
    'best_params'    : {k: (v if not isinstance(v, float) else round(v, 8))
                        for k, v in best_p.items()},
    'cv_auc'         : round(cv_auc, 6),
    'train_auc'      : round(train_auc, 6),
    'n_trials'       : N_TRIALS,
    'n_folds'        : N_FOLDS,
    'n_train'        : int(X.shape[0]),
    'n_features'     : int(X.shape[1]),
    'feature_cols'   : feature_cols,
    'native_nan'     : NATIVE_NAN,
    'sklearn_version': sklearn.__version__,
    'timestamp'      : pd.Timestamp.now().isoformat()
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

trials_df = study_rf.trials_dataframe()
trials_df.to_csv(trials_path, index=False)

print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
print(f'  {model_path.name:<45} ({model_path.stat().st_size/1e6:.2f} MB)')
print(f'  {meta_path.name}')
print(f'  {trials_path.name}')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
print(f'    - {model_path.name}')
print(f'    - {meta_path.name}')
print('\n>>> Luego correr localmente: rfr_cloud_predict.ipynb')

## Resumen Final — Random Forest Cloud

In [ ]:
import platform
print('=' * 65)
print('RANDOM FOREST CLOUD — RESUMEN FINAL')
print('=' * 65)
print(f'  CV AUC ({N_FOLDS}-fold OOF) : {cv_auc:.5f}')
print(f'  Train AUC (in-sample) : {train_auc:.5f}')
print(f'  Gap                   : {abs(train_auc - cv_auc):.5f}')
print(f'  n_train               : {X.shape[0]:,}')
print('─' * 65)
print(f'  Mejores hiperparametros:')
for k, v in best_p.items():
    print(f'    {k:<22}: {v}')
print('─' * 65)
print(f'  sklearn   : {sklearn.__version__}')
print(f'  optuna    : {optuna.__version__}')
print(f'  numpy     : {np.__version__}')
print(f'  Python    : {platform.python_version()}')
print(f'  Timestamp : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)